# Silver — Olist Marketing Funnel e IPCA mensal

Lê Bronze Delta; MQL filtradas de 01/06/2017 a 31/12/2017 pela data do primeiro contacto; Closed Deals ligados por `mql_id` sem filtrar `won_date` (preserva conversões posteriores); IPCA independente por mês. Grava apenas `silver.marketing_funnel` e `silver.ipca_mensal`. Não cria Gold.

In [ ]:
DATA_INICIO = "2017-06-01"
DATA_FIM = "2017-12-31"

In [ ]:
from pyspark.sql import SparkSession, functions as F
from delta import configure_spark_with_delta_pip
HDFS = "hdfs://hdfs-nn:9000"
WAREHOUSE = f"{HDFS}/warehouse"
builder = (SparkSession.builder.appName("EDSTD Bruno Marketing")
    .config("spark.sql.warehouse.dir", WAREHOUSE)
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0")
    .enableHiveSupport())
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS silver LOCATION '{WAREHOUSE}/silver.db/'")
mql_raw = spark.table("bronze.bronze_mql")
deals_raw = spark.table("bronze.bronze_closed_deals")
ipca_raw = spark.table("bronze.bronze_ipca")
for nome, df in (("MQL", mql_raw), ("Closed Deals", deals_raw), ("IPCA", ipca_raw)):
    print(nome, df.count(), "linhas")
    df.printSchema()

## 1. Limpeza e filtro MQL

In [ ]:
mql = (mql_raw
    .withColumn("mql_id", F.trim("mql_id"))
    .withColumn("first_contact_date", F.to_date("first_contact_date", "yyyy-MM-dd"))
    .withColumn("origin", F.when(F.col("origin").isNull() | (F.trim("origin") == ""), "unknown")
                .otherwise(F.lower(F.trim("origin"))))
    .withColumn("contact_month", F.trunc("first_contact_date", "month"))
    .filter(F.col("first_contact_date").between(DATA_INICIO, DATA_FIM))
    .select("mql_id", "first_contact_date", "contact_month", "landing_page_id", "origin"))
assert mql.filter(F.col("mql_id").isNull() | (F.col("mql_id") == "")).count() == 0
assert mql.groupBy("mql_id").count().filter("count > 1").count() == 0
print("MQL no período:", mql.count())

## 2. Closed Deals — Boolean e tipos

In [ ]:
def para_booleano(coluna):
    v = F.lower(F.trim(F.col(coluna)))
    return (F.when(v.isin("true", "1", "yes", "y", "sim", "s"), F.lit(True))
             .when(v.isin("false", "0", "no", "n", "não", "nao"), F.lit(False))
             .otherwise(F.lit(None).cast("boolean")))
for coluna in ("has_company", "has_gtin"):
    print("Valores originais:", coluna)
    deals_raw.groupBy(coluna).count().orderBy(F.desc("count")).show(30, truncate=False)
    v = F.lower(F.trim(F.col(coluna)))
    invalidos = deals_raw.filter(F.col(coluna).isNotNull() & (v != "") &
       ~v.isin("true", "false", "1", "0", "yes", "no", "y", "n", "sim", "s", "não", "nao"))
    assert invalidos.count() == 0, f"Valor booleano inesperado em {coluna}"
deals = (deals_raw
    .withColumn("mql_id", F.trim("mql_id"))
    .withColumn("seller_id", F.trim("seller_id"))
    .withColumn("won_date", F.to_timestamp("won_date", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("declared_monthly_revenue", F.col("declared_monthly_revenue").cast("double"))
    .withColumn("declared_product_catalog_size", F.col("declared_product_catalog_size").cast("integer"))
    .withColumn("has_company", para_booleano("has_company"))
    .withColumn("has_gtin", para_booleano("has_gtin"))
    .withColumn("average_stock", F.when(F.col("average_stock").isNull() |
         (F.trim("average_stock") == "") | (F.lower(F.trim("average_stock")) == "unknown"),
         F.lit(None).cast("string")).otherwise(F.trim("average_stock")))
    .drop("_data_ingestao", "_ficheiro_origem"))
assert deals.filter(F.col("mql_id").isNull() | (F.col("mql_id") == "")).count() == 0
assert deals.groupBy("mql_id").count().filter("count > 1").count() == 0
print("Closed Deals:", deals.count())

## 3. Integração das leads

In [ ]:
marketing = (mql.join(deals.withColumn("converted", F.lit(1)), "mql_id", "left")
    .withColumn("converted", F.coalesce(F.col("converted"), F.lit(0))))
assert marketing.count() == mql.count(), "JOIN alterou número de leads"
assert marketing.groupBy("mql_id").count().filter("count > 1").count() == 0
print("Leads:", marketing.count())
print("Convertidas:", marketing.filter("converted = 1").count())
marketing.select("mql_id", "first_contact_date", "origin", "seller_id", "converted").show(5, truncate=False)

## 4. IPCA mensal — dimensão futura

In [ ]:
ipca = (ipca_raw
    .withColumn("reference_month", F.to_date(F.concat(F.lit("01/"), F.col("Date")), "dd/MM/yyyy"))
    .withColumn("ipca_rate_12m", F.col("IPCA Rate (last 12 months)").cast("double"))
    .select("reference_month", "ipca_rate_12m")
    .filter(F.col("reference_month").between(DATA_INICIO, DATA_FIM)))
assert ipca.filter(F.col("reference_month").isNull() | F.col("ipca_rate_12m").isNull()).count() == 0
assert ipca.groupBy("reference_month").count().filter("count > 1").count() == 0
assert ipca.count() == 7, "Esperados sete meses entre junho e dezembro de 2017"
ipca.orderBy("reference_month").show(truncate=False)

## 5. Escrita Delta e validação

In [ ]:
def gravar_silver(df, tabela):
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .option("path", f"{WAREHOUSE}/silver.db/{tabela}/")
       .saveAsTable(f"silver.{tabela}"))
    print(f"silver.{tabela}: {spark.table(f'silver.{tabela}').count()} linhas")
gravar_silver(marketing, "marketing_funnel")
gravar_silver(ipca, "ipca_mensal")

In [ ]:
mf = spark.table("silver.marketing_funnel")
im = spark.table("silver.ipca_mensal")
assert mf.count() == mql.count()
assert mf.select("mql_id").distinct().count() == mf.count()
assert im.count() == 7
assert mf.schema["has_company"].dataType.simpleString() == "boolean"
assert mf.schema["has_gtin"].dataType.simpleString() == "boolean"
print("Convertidas sem seller_id:", mf.filter("converted = 1 AND seller_id IS NULL").count())
mf.agg(F.count("*").alias("leads"), F.sum("converted").alias("convertidas"),
    F.round(F.avg(F.col("converted").cast("double")) * 100, 2).alias("taxa_conversao_pct")).show()
spark.sql("SHOW TABLES IN silver").show(truncate=False)
for t in ("marketing_funnel", "ipca_mensal"):
    spark.sql(f"DESCRIBE DETAIL silver.{t}").select("format", "location", "numFiles").show(truncate=False)

In [ ]:
spark.stop()